# 🫁 Lung Lesion Segmentation - Kaggle Training & Evaluation Pipeline

Notebook này được thiết kế để huấn luyện và đánh giá các mô hình phân đoạn tổn thương CT phổi:
1. **Attention U-Net** (Mô hình đề xuất - Proposed Model)
2. **Standard U-Net** (Baseline đối chuẩn 1)
3. **SegNet** (Baseline đối chuẩn 2)

> **Yêu cầu môi trường Kaggle:**
>- **Accelerator:** GPU (NVIDIA P100 hoặc T4 x2)
>- **Internet:** On (để cài đặt `nibabel`)
>- **Input Dataset:** Mount bộ dữ liệu CT NIfTI (Coronacases + Radiopaedia) vào `/kaggle/input`

## 1. Cài đặt thư viện phụ thuộc

In [ ]:
# Cài đặt thư viện cần thiết (Kaggle đã có sẵn torch, albumentations, cv2, pyyaml)
!pip install -q nibabel

## 2. Kiểm tra phần cứng GPU & Môi trường

In [ ]:
import torch
import os
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Device Count   : {torch.cuda.device_count()}")
else:
    print("CẢNH BÁO: Chưa bật GPU Accelerator trên Kaggle! Hãy vào Settings -> Accelerator -> Chọn GPU P100 hoặc T4.")

## 3. Xác định vị trí Dataset trên Kaggle
Cell này tự động quét `/kaggle/input` để tìm thư mục chứa các file CT NIfTI (`.nii` hoặc `.nii.gz`).

In [ ]:
kaggle_input = Path("/kaggle/input")
raw_data_dir = None

if kaggle_input.exists():
    print("Các thư mục trong /kaggle/input:")
    for p in kaggle_input.iterdir():
        print(f" - {p.name}")
        # Tìm folder chứa ct_volumes hoặc file nii
        if (p / "ct_volumes").exists():
            raw_data_dir = str(p)
        elif any(p.glob("*.nii*")):
            raw_data_dir = str(p)

if raw_data_dir:
    os.environ["RAW_DATA_DIR"] = raw_data_dir
    print(f"\n[+] Tự động phát hiện raw data tại: {raw_data_dir}")
else:
    print("\n[!] Chưa tự động tìm thấy folder nii. Bạn có thể tự gán os.environ['RAW_DATA_DIR'] = '/kaggle/input/<ten-dataset>'")

## 4. Chạy Unit Tests kiểm tra tính toàn vẹn (Sanity Check)

In [ ]:
!python tests/test_dice_loss.py
!python tests/test_hybrid_loss.py
!python tests/test_segnet.py
!python tests/test_unet.py
!python tests/test_pipeline.py

## 5. Huấn luyện Mô hình Attention U-Net (Đề tài chính)
File cấu hình: `configs/attention_unet.yaml` (50 epochs, AdamW, CosineAnnealingLR, Soft Dice + BCE Hybrid Loss).

In [ ]:
!python train.py --config configs/attention_unet.yaml

## 6. Huấn luyện Mô hình Baseline 1: Standard U-Net
File cấu hình: `configs/baseline_unet.yaml`.

In [ ]:
!python train.py --config configs/baseline_unet.yaml

## 7. Huấn luyện Mô hình Baseline 2: SegNet
File cấu hình: `configs/segnet.yaml`.

In [ ]:
!python train.py --config configs/segnet.yaml

## 8. Đánh giá kiểm thử trên Independent Test Set (15% Patient-level)
Xuất 2 bảng CSV theo chuẩn báo cáo nghiên cứu:
- `reports/test_metrics_slice_level.csv`
- `reports/test_metrics_patient_level.csv`

In [ ]:
# Đánh giá Attention U-Net
!python evaluate.py --checkpoint checkpoints/EXP-ATTUNET-001_best.pth --model AttentionUNet --output_dir reports/attention_unet

# Đánh giá Standard U-Net
!python evaluate.py --checkpoint checkpoints/EXP-UNET-001_best.pth --model UNet --output_dir reports/unet

# Đánh giá SegNet
!python evaluate.py --checkpoint checkpoints/EXP-SEGNET-001_best.pth --model SegNet --output_dir reports/segnet

## 9. Kiểm tra file kết quả & Checkpoints đã tạo

In [ ]:
print("Checkpoints đã lưu:")
!ls -lh checkpoints/

print("\nBáo cáo kết quả:")
!ls -lh reports/*